# KIT RAG（sarashina2.2-3b）実行ノートブック

| 段階 | セル | 内容 |
|---|---|---|
| 準備 | 0〜2 | GPU確認、コード取得、保存データの復元 |
| 索引づくり | 3〜4 | クロール → 前処理・索引作成（保存データがあれば自動で飛ばす） |
| 動作確認 | 5〜6 | モデル読み込みと試し回答、検索結果の確認 |
| 本番 | 7 | 質問JSONの全問に回答 |

**実行環境**：ブラウザのColab（ランタイムのタイプ → T4 GPU）、またはVS CodeのColab拡張（New Colab Server → GPU → T4）。

**2回目以降**：保存データがあればセル0 → 1 → 2 → 5 → 6・7 だけで済む。

**コードを直したとき**：VS Codeでプッシュ → カーネルを再起動 → セル1から（`git pull`で最新版になる）。

**チャンクの大きさや埋め込みモデルを変えたとき**：セル2の直後に `RESTORED = False` を実行してからセル3・4へ。

## 準備

In [1]:
# 0. GPUの確認（「Tesla T4」と表示されればOK）
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

Tesla T4, 15360 MiB


In [1]:
# 1. GitHubからコードを取得（2回目以降は最新版に更新）
GITHUB_USER = "miyamoto-gt"
REPO = "KIT_LLM_RAG"

import os
REPO_DIR = f"/content/{REPO}"
ENV_PATHS = ["/content/.env", "/content/drive/MyDrive/.env"]   # .env を探す場所（上から順）

def load_env(paths):
    """KEY=VALUE 形式の .env を読み、環境変数に入れる（ライブラリ不要の簡易版）"""
    for p in paths:
        if os.path.exists(p):
            for line in open(p, encoding="utf-8"):
                line = line.strip()
                if line and not line.startswith("#") and "=" in line:
                    k, v = line.split("=", 1)
                    os.environ.setdefault(k.strip(), v.strip().strip('"').strip("'"))
            print(".env を読み込みました:", p)
            return True
    return False

def get_token():
    load_env(ENV_PATHS)
    if os.environ.get("GITHUB_TOKEN"):          # ① .env
        return os.environ["GITHUB_TOKEN"]
    try:                                         # ② ブラウザのColab：シークレット
        from google.colab import userdata
        return userdata.get("GITHUB_TOKEN")
    except Exception:                            # ③ その場で入力
        from getpass import getpass
        return getpass("GitHubのトークンを貼り付けてEnter: ")

if not os.path.exists(REPO_DIR):
    token = get_token()
    print("取得先:", f"github.com/{GITHUB_USER}/{REPO}")
    !git clone https://{token}@github.com/{GITHUB_USER}/{REPO}.git {REPO_DIR}
    del token
    assert os.path.exists(REPO_DIR), "クローン失敗：上のメッセージを確認"
os.chdir(REPO_DIR)
!git pull -q
!git log --oneline -1
%pip install -q -r requirements.txt

取得先: github.com/miyamoto-gt/KIT_LLM_RAG
Cloning into '/content/KIT_LLM_RAG'...
remote: Enumerating objects: 23, done.
remote: Counting objects: 100% (23/23), done.
remote: Compressing objects: 100% (20/20), done.
remote: Total 23 (delta 0), reused 20 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (23/23), 19.28 KiB | 9.64 MiB/s, done.
ad07b15 (HEAD -> main, origin/main, origin/HEAD) base
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.4/46.4 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 158.7/158.7 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 409.6/409.6 kB 26.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 115.0 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 257.6/257.6 kB 40.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 837.9/837.9 kB 86.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 159.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━

In [3]:
# 2. 保存データの復元（Driveが使えない環境では、毎回作り直すモードになる）
SAVE = None
try:
    from google.colab import drive
    drive.mount("/content/drive")
    SAVE = "/content/drive/MyDrive/kit_rag_data"
    os.makedirs(SAVE, exist_ok=True)
except Exception as e:
    print("Driveは使えません（毎回作り直します）:", type(e).__name__)

RESTORED = bool(SAVE) and os.path.exists(f"{SAVE}/index/faiss.index")
if RESTORED:
    !cp -r {SAVE}/raw {SAVE}/index {SAVE}/chunks.jsonl data/
print("保存データを復元しました（セル3・4は飛ばします）" if RESTORED else "保存データなし：セル3・4で作ります")

Mounted at /content/drive
保存データなし：セル3・4で作ります


## 索引づくり

In [4]:
# 3. ① 収集（1秒間隔なので数十分）
if not RESTORED:
    !python crawl.py
    !echo "保存ページ数:" && wc -l < data/raw/manifest.jsonl

Traceback (most recent call last):
  File "/content/KIT_LLM_RAG/crawl.py", line 16, in <module>
    from KIT_LLM_RAG.config import CFG
ModuleNotFoundError: No module named 'KIT_LLM_RAG'
保存ページ数:
/bin/bash: line 1: data/raw/manifest.jsonl: No such file or directory


In [ ]:
# 4. ②③ 前処理・チャンク化 → ④ ベクトル索引 →（Driveが使えれば）保存
if not RESTORED:
    !python preprocess.py
    !python index.py
    if SAVE:
        !cp -r data/raw data/index data/chunks.jsonl {SAVE}/
        print("Driveに保存しました")

## 動作確認

In [ ]:
# 5. モデルを1回だけ読み込み、試しに答えさせる（読み込みに数分）
from pipeline import RAGPipeline
rag = RAGPipeline()

def ask(q):
    r = rag.answer(q)
    print(f"[{r['qtype']}] {q}\n  → {r['answer']}  (top_score={r['top_score']:.3f})\n  sources={r['sources'][:2]}\n")

for q in ["図書館は何号館にありますか？",
          "コンビニは何号館の何階にありますか？",
          "夢考房とはどのような施設ですか？"]:
    ask(q)

In [ ]:
# 6. 検索だけを確認（回答が外れたとき、検索と生成のどちらが原因か切り分ける）
Q = "図書館は何号館にありますか？"   # ← 調べたい質問に変える
for chunk, score in rag.retriever.search(Q):
    print(f"{score:.3f}", chunk.text[:80].replace("\n", " "))

## 本番

In [ ]:
# 7. 質問JSONの全問に回答（キー名は run_eval.py の TODO を合わせる）
import json, time, run_eval
qs = run_eval.load_questions("questions.json")   # URLでも可
results, log = [], []
for i, item in enumerate(qs):
    t = time.time()
    r = rag.answer(item[run_eval.QUESTION_KEY])
    results.append({run_eval.ID_KEY: item.get(run_eval.ID_KEY, i), "answer": r["answer"]})
    log.append({**r, "sec": round(time.time() - t, 2)})
    print(f"[{i}] ({r['qtype']}) {r['question']} -> {r['answer']}")
json.dump(results, open("answers.json", "w", encoding="utf-8"), ensure_ascii=False, indent=2)
json.dump(log, open("answers_log.json", "w", encoding="utf-8"), ensure_ascii=False, indent=2)
print("saved answers.json / answers_log.json")